In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader

from pathlib import Path
from decouple import AutoConfig

/var/folders/f5/jv7k5yw97479jf4xr73rgvd40000gn/T/ipykernel_52856/3254719189.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
cwd = Path.cwd()
project_dir = cwd / "langchain" if (cwd / "langchain").is_dir() else cwd
config = AutoConfig(search_path=str(project_dir))
api_key = config("OPENAI_API_KEY")

In [10]:
openai = ChatOpenAI(api_key=api_key, model_name="gpt-4o-mini", store=False, temperature=0)

In [5]:
loader = TextLoader('base_conhecimento_britadeira.txt')
documents = loader.load()

In [ ]:
historico_conversas = """
Cliente: Minha britadeira não liga.
Chatbot: Você já verificou se a bateria está carregada e conectada corretamente?
"""
pergunta = "Minha britadeira não liga. Eu já verifiquei se a bateria está carregada e conectada corretamente."

In [7]:
inputs = {
    "context": "\n".join([doc.page_content for doc in documents]),
    "question": pergunta,
    "history": historico_conversas
}

In [8]:
prompt_base_conhecimento = PromptTemplate(
    input_variables=["context", "question", "history"],
    template="""Use o seguinte contexto para responder à pergunta.
    Responda apenas com base nas informações fornecidas.
    Não forneça instruções sobre procedimentos já realizados.
    Não utilize informações externas ao contexto.
    Contexto: {context}
    Pergunta: {question}"""
)
prompt_historico_conversas = PromptTemplate(
    input_variables=["history", "question"],
    template="""Use o seguinte histórico de conversas para responder à pergunta.
    Responda apenas com base nas informações fornecidas.
    Não forneça instruções sobre procedimentos já realizados.
    Não utilize informações externas ao contexto.
    Histórico de conversas: {history}
    Pergunta: {question}"""
)
prompt_final = PromptTemplate(
    input_variables=["resposta_base_conhecimento", "resposta_historico_conversas"],
    template="""Combine as respostas a seguir para gerar uma resposta final.
    Não forneça instruções sobre procedimentos já realizados.
    Resposta da base de conhecimento: {resposta_base_conhecimento}
    Resposta do histórico de conversas: {resposta_historico_conversas}"""
)

In [9]:
print(prompt_base_conhecimento)

input_variables=['context', 'question'] input_types={} partial_variables={} template='Use o seguinte contexto para responder à pergunta.\n    Responda apenas com base nas informações fornecidas.\n    Não forneça instruções sobre procedimentos já realizados.\n    Não utilize informações externas ao contexto.\n    Contexto: {context}\n    Pergunta: {question}'


In [11]:
chain_base_conhecimento = prompt_base_conhecimento | openai
chain_historico_conversas = prompt_historico_conversas | openai
chain_final = prompt_final | openai

In [12]:
print(chain_base_conhecimento)

first=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Use o seguinte contexto para responder à pergunta.\n    Responda apenas com base nas informações fornecidas.\n    Não forneça instruções sobre procedimentos já realizados.\n    Não utilize informações externas ao contexto.\n    Contexto: {context}\n    Pergunta: {question}') middle=[] last=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.7.1', 'langchain': '1.4.7', 'langchain-openai': '1.7.3'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature

In [13]:
print(chain_historico_conversas)

first=PromptTemplate(input_variables=['history', 'question'], input_types={}, partial_variables={}, template='Use o seguinte histórico de conversas para responder à pergunta.\n    Responda apenas com base nas informações fornecidas.\n    Não forneça instruções sobre procedimentos já realizados.\n    Não utilize informações externas ao contexto.\n    Histórico de conversas: {history}\n    Pergunta: {question}') middle=[] last=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.7.1', 'langchain': '1.4.7', 'langchain-openai': '1.7.3'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'att

In [14]:
resultado_base_conhecimento = chain_base_conhecimento.invoke(
    {
        "context": inputs["context"],
        "question": inputs["question"],
    }
)
resultado_historico_conversas = chain_historico_conversas.invoke(
    {
        "history": inputs["history"],
        "question": inputs["question"],
    }
)
resultado_final = chain_final.invoke(
    {
        "resposta_base_conhecimento": resultado_base_conhecimento,
        "resposta_historico_conversas": resultado_historico_conversas,
    }
)

In [15]:
print(f"Resultado Base de Conhecimento:\n {resultado_base_conhecimento}")
print("-----")
print(f"Resultado Histórico de Conversas:\n {resultado_historico_conversas}")


Resultado Base de Conhecimento:
 content='Aguarde 30 segundos após conectar a bateria para que o sistema inicialize. Verifique se o interruptor liga/desliga está na posição correta. Se o problema persistir, contate o suporte técnico pelo 0800 555 5555.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 681, 'total_tokens': 733, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a61d6793b9', 'id': 'chatcmpl-Efji08eHq3LsYmN9TmsKVgFAzdfrz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a19bd6-0ebf-7379-b254-fb013c216eb4-0' tool_calls=[] invalid_tool_calls

In [16]:
print(resultado_final.content)

Aguarde 30 segundos após conectar a bateria para que o sistema inicialize e verifique se o interruptor liga/desliga está na posição correta. Além disso, você já tentou verificar se há algum problema com o botão de liga/desliga ou se há algum fusível queimado? Se o problema persistir, contate o suporte técnico pelo 0800 555 5555.


In [17]:
print(resultado_final)

content='Aguarde 30 segundos após conectar a bateria para que o sistema inicialize e verifique se o interruptor liga/desliga está na posição correta. Além disso, você já tentou verificar se há algum problema com o botão de liga/desliga ou se há algum fusível queimado? Se o problema persistir, contate o suporte técnico pelo 0800 555 5555.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 731, 'total_tokens': 810, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e327275518', 'id': 'chatcmpl-Efji2XnWjm8kR4nswrQoXsoc6RlRA', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': Non